# 03 — Generate + validate a plan

Loads a trained checkpoint, asks it to generate a plan for a sample plot + room list + adjacency graph you specify below, then runs the same kind of validity check the product's deterministic engine is unit-tested against (no overlaps, everything in bounds) — see `tests/unit/plan-generation-engine.test.ts` in the main app for the TypeScript equivalent.

Run `02_train_baseline.ipynb` first — you need a checkpoint to load here.

In [ ]:
GITHUB_REPO_URL = ""  # e.g. "https://github.com/<you>/planHouse.git"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/planHouse-ml/floor-plan-gen"  # used if GITHUB_REPO_URL is empty

import os
import sys

from google.colab import drive
drive.mount("/content/drive")  # needed either way, to reach the checkpoint

if GITHUB_REPO_URL:
    if not os.path.isdir("/content/planHouse"):
        !git clone {GITHUB_REPO_URL} /content/planHouse
    else:
        !git -C /content/planHouse pull  # already cloned earlier this session — just update
    PROJECT_DIR = "/content/planHouse/ml/floor-plan-gen"
else:
    PROJECT_DIR = DRIVE_PROJECT_PATH
    assert os.path.isdir(PROJECT_DIR), (
        f"{PROJECT_DIR} not found — upload ml/floor-plan-gen there first, "
        f"or set GITHUB_REPO_URL above instead."
    )

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
%pip install -q -r {os.path.join(PROJECT_DIR, "requirements.txt")}

In [ ]:
from floorplan_gen.generate import load_model

CHECKPOINT_PATH = "/content/drive/MyDrive/planHouse-ml/floor-plan-gen/checkpoints/best.pt"
model = load_model(CHECKPOINT_PATH)

## Define a plot + room list

A 20 x 40 (units don't matter to the model — metres or feet both work identically; they only matter for how you read the output afterwards) rectangular plot with a living room, two bedrooms, a kitchen, and a bathroom, wired roughly the way a real home's connectivity would look: the living room touches everything, the bathroom connects via a door specifically. Change this to try other plots/room lists/graphs.

In [ ]:
from floorplan_gen.data.resplan_dataset import build_query_sample
from floorplan_gen.generate import generate_plan

boundary = [(0, 0), (20, 0), (20, 40), (0, 40)]
room_types = ["living", "bedroom", "bedroom", "kitchen", "bathroom"]
edges = [
    (0, 1, "adjacency"),
    (0, 2, "adjacency"),
    (0, 3, "adjacency"),
    (0, 4, "via_door"),
]

sample = build_query_sample(boundary, room_types, edges)
plan = generate_plan(model, sample)

for room in plan.rooms:
    print(f"{room.name:12s} x={room.x:6.2f} y={room.y:6.2f} w={room.width:6.2f} h={room.height:6.2f}")
if plan.warnings:
    print("\nWarnings:")
    for w in plan.warnings:
        print(" -", w)

In [ ]:
from floorplan_gen.visualize import plot_generated_plan

plot_generated_plan(plan, boundary_coords=boundary, title="Generated plan (v1 baseline)")

## Validity check

Same invariants the product's deterministic engine is unit-tested against: every room inside the boundary, no two rooms overlapping. `generate_plan`'s cleanup pass is supposed to guarantee this — this cell is the actual proof, not an assumption.

In [ ]:
def check_validity(plan, boundary):
    xs = [c[0] for c in boundary]
    ys = [c[1] for c in boundary]
    minx, maxx, miny, maxy = min(xs), max(xs), min(ys), max(ys)
    issues = []

    for r in plan.rooms:
        out_of_bounds = (
            r.x < minx - 1e-3
            or r.y < miny - 1e-3
            or r.x + r.width > maxx + 1e-3
            or r.y + r.height > maxy + 1e-3
        )
        if out_of_bounds:
            issues.append(f"{r.name} is out of bounds")

    for i in range(len(plan.rooms)):
        for j in range(i + 1, len(plan.rooms)):
            a, b = plan.rooms[i], plan.rooms[j]
            overlaps = a.x < b.x + b.width and a.x + a.width > b.x and a.y < b.y + b.height and a.y + a.height > b.y
            if overlaps:
                issues.append(f"{a.name} overlaps {b.name}")

    return issues


issues = check_validity(plan, boundary)
print("VALID — no overlaps, everything in bounds" if not issues else "\n".join(issues))